# Treinamento ponta a ponta e persistência de modelo

Este notebook reúne as etapas estudadas até aqui em um único fluxo: dados tabulares mistos, divisão treino/teste, imputação, codificação, escalonamento, busca de hiperparâmetros, avaliação e salvamento do modelo. O resultado é um único artefato `.pickle` capaz de receber os dados brutos e produzir previsões.

Usamos o dataset Breast Cancer Wisconsin, já apresentado no módulo de classificação e incluído no scikit-learn. Como ele contém apenas variáveis numéricas, transformamos uma delas em uma faixa categórica somente para praticar o encoder.

## Imports e configuração

`Pipeline` garante que cada transformação seja ajustada exclusivamente nos dados de treino de cada dobra da validação cruzada. Isso evita *data leakage*.

In [ ]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_breast_cancer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    f1_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

## Dados tabulares com ausências

O dataset contém medidas numéricas de tumores. A coluna `faixa_raio`, derivada de `mean radius` com intervalos fixos, é a feature categórica do exemplo. Introduzimos algumas ausências propositalmente para demonstrar os imputers. O alvo `maligno` segue a mesma codificação do módulo de classificação: 1 para maligno e 0 para benigno.

In [ ]:
cancer = load_breast_cancer(as_frame=True)
dados = cancer.data.copy()

# Intervalos fixos: em dados reais, devem ser definidos pelo domínio.
dados['faixa_raio'] = pd.cut(
    dados.pop('mean radius'),
    bins=[-np.inf, 12, 18, np.inf],
    labels=['pequeno', 'medio', 'grande'],
).astype(object)
dados['maligno'] = (cancer.target == 0).astype(int)

for coluna in ['mean texture', 'worst area']:
    indices = rng.choice(dados.index, size=30, replace=False)
    dados.loc[indices, coluna] = np.nan

indices = rng.choice(dados.index, size=20, replace=False)
dados.loc[indices, 'faixa_raio'] = np.nan

dados.head()

In [ ]:
dados.info()
dados.isna().sum().sort_values(ascending=False)

## Separação treino/teste

O teste é separado antes de qualquer ajuste e só aparece na avaliação final. Como o alvo é categórico, `stratify=y` preserva sua proporção nas duas partes.

In [ ]:
X = dados.drop(columns='maligno')
y = dados['maligno']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

print(f'Treino: {X_train.shape[0]} linhas | Teste: {X_test.shape[0]} linhas')
print(y.value_counts(normalize=True).rename('proporcao'))

## Pré-processamento e pipeline

Colunas numéricas recebem imputação pela mediana e `StandardScaler`. Colunas categóricas recebem a categoria mais frequente e `OneHotEncoder`. `handle_unknown='ignore'` permite prever uma categoria que não existia no treino sem falhar.

Ao colocar o pré-processamento dentro do pipeline, o `RandomizedSearchCV` executa essas etapas corretamente em cada dobra.

In [ ]:
colunas_numericas = X.select_dtypes(include='number').columns.tolist()
colunas_categoricas = ['faixa_raio']

pipeline_numerico = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

pipeline_categorico = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])

preprocessador = ColumnTransformer([
    ('numericas', pipeline_numerico, colunas_numericas),
    ('categoricas', pipeline_categorico, colunas_categoricas),
])

pipeline_modelo = Pipeline([
    ('preprocessador', preprocessador),
    ('modelo', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
])

pipeline_modelo

## Busca de hiperparâmetros

O `RandomizedSearchCV` avalia 48 combinações de um espaço amplo do `RandomForestClassifier`, incluindo número e profundidade das árvores, amostragem de atributos, regularização e pesos de classe. Usamos `recall` para a classe positiva (`maligno = 1`): essa métrica privilegia a identificação dos tumores malignos e reduz falsos negativos, que são o erro mais caro neste contexto. A busca é feita somente em `X_train` e `y_train`.

In [ ]:
distribuicoes_parametros = {
    'modelo__n_estimators': np.arange(300, 1_201, 100),
    'modelo__max_depth': [None, 5, 10, 20, 30, 50],
    'modelo__min_samples_split': np.arange(2, 21),
    'modelo__min_samples_leaf': np.arange(1, 11),
    'modelo__max_features': ['sqrt', 'log2', 0.3, 0.5, 0.7, 1.0],
    'modelo__criterion': ['gini', 'entropy', 'log_loss'],
    'modelo__class_weight': [None, 'balanced', 'balanced_subsample'],
    'modelo__ccp_alpha': np.logspace(-5, -1, 25),
}
validacao = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

busca = RandomizedSearchCV(
    estimator=pipeline_modelo,
    param_distributions=distribuicoes_parametros,
    n_iter=48,
    scoring='recall',
    cv=validacao,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    refit=True,
)
busca.fit(X_train, y_train)

print(f'Melhores parâmetros: {busca.best_params_}')
print(f'Melhor recall médio na validação: {busca.best_score_:.3f}')

## Avaliação final

Agora, e somente agora, avaliamos o melhor pipeline no conjunto de teste que ficou isolado.

In [ ]:
melhor_modelo = busca.best_estimator_
predicoes = melhor_modelo.predict(X_test)
probabilidades = melhor_modelo.predict_proba(X_test)[:, 1]

print(f'Recall (sensibilidade) para maligno no teste: {recall_score(y_test, predicoes):.3f}')
print(f'F1 no teste (métrica complementar): {f1_score(y_test, predicoes):.3f}')
print(f'ROC-AUC no teste: {roc_auc_score(y_test, probabilidades):.3f}')
print('\nRelatório de classificação:')
print(classification_report(y_test, predicoes, target_names=['benigno', 'maligno']))

ConfusionMatrixDisplay.from_predictions(y_test, predicoes, display_labels=['benigno', 'maligno']);

## Persistência em `.pickle`

Salvamos o **pipeline inteiro**, não apenas a regressão logística. Assim, no uso futuro, os mesmos imputers, encoder e scaler são aplicados antes da previsão. O diretório `artifacts/` é criado automaticamente.

Nunca carregue arquivos pickle de fonte não confiável: desserializá-los pode executar código. Para reproduzir o modelo, registre também as versões das bibliotecas usadas.

In [ ]:
raiz_projeto = Path.cwd().resolve()
if raiz_projeto.name == 'notebooks':
    raiz_projeto = raiz_projeto.parent

caminho_modelo = raiz_projeto / 'artifacts/model.pickle'
caminho_modelo.parent.mkdir(parents=True, exist_ok=True)

with caminho_modelo.open('wb') as arquivo:
    pickle.dump(melhor_modelo, arquivo)

print(f'Modelo salvo em: {caminho_modelo.resolve()}')